# ASL-to-English Translation Training on Kaggle

This notebook clones our landmark-based ASL translation repository, installs dependencies, verifies the coordinate dataset, and starts training using Kaggle GPUs.

## 1. Setup and Installation

In [ ]:
# Clone or update the repository
import os
if not os.path.exists('gloss-free-asl-translation'):
    !git clone https://github.com/yyouretoast/gloss-free-asl-translation.git
    %cd /kaggle/working/gloss-free-asl-translation
else:
    %cd /kaggle/working/gloss-free-asl-translation
    !git pull


In [ ]:
# Filter out torch/torchvision to keep Kaggle's GPU-optimized pre-installs
!sed -i '/torch/d' requirements.txt
!pip install -r requirements.txt


## 2. Environment Configuration (Optional)

Set your Hugging Face token to enable faster downloads of the `t5-small` model.

In [ ]:
import os
# Set your HF Token if available
# os.environ["HF_TOKEN"] = "your_huggingface_token_here"

## 3. Real-World Dataset Validation

Before training, run the profile validation script to inspect the tracking dropouts and sequence lengths of the coordinates.

In [ ]:
# Run dataset validation profile (point --data_dir to the actual coordinates directory)
# For YouTube-ASL:
# !python -m src.validate_dataset --data_dir /kaggle/input/youtube-asl/landmarks

# For How2Sign (auto-resolving paths):
import os
data_dir = next((p for p in [
    '/kaggle/input/datasets/nazarboholii/how2sign/train_2D_keypoints/openpose_output/json',
    '/kaggle/input/how2sign/train_2D_keypoints/openpose_output/json',
    '/kaggle/input/how2sign-keypoints/train_2D_keypoints/openpose_output/json',
    '/kaggle/input/datasets/nazarboholii/how2sign-keypoints/train_2D_keypoints/openpose_output/json'
] if os.path.exists(p)), '/kaggle/input/datasets/nazarboholii/how2sign/train_2D_keypoints/openpose_output/json')
!python -m src.validate_dataset --data_dir {data_dir}


In [ ]:
# Dataset structure audit utility (adds flexibility for How2Sign or custom data)
import os
import glob
import numpy as np

# Choose your input dataset path (auto-resolving paths)
for path in [
    '/kaggle/input/datasets/nazarboholii/how2sign',
    '/kaggle/input/how2sign',
    '/kaggle/input/how2sign-keypoints',
    '/kaggle/input/datasets/nazarboholii/how2sign-keypoints'
]:
    if os.path.exists(path):
        input_dir = path
        break
else:
    input_dir = '/kaggle/input/datasets/nazarboholii/how2sign'

# Fast inspection without slow recursive globbing
npz_files = []
npy_files = []
json_dirs = []

# Check candidate OpenPose directory directly
json_cand = os.path.join(input_dir, "train_2D_keypoints/openpose_output/json")
if os.path.exists(json_cand):
    # list directories under this path
    json_dirs = [os.path.join(json_cand, d) for d in os.listdir(json_cand) if os.path.isdir(os.path.join(json_cand, d))]

# If no json dirs, check if there are NPZs in input_dir directly
if not json_dirs:
    if os.path.exists(input_dir):
        # Check direct files
        npz_files = [os.path.join(input_dir, f) for f in os.listdir(input_dir) if f.endswith('.npz')]
        npy_files = [os.path.join(input_dir, f) for f in os.listdir(input_dir) if f.endswith('.npy')]

print(f"Total .npz files found: {len(npz_files)}")
print(f"Total .npy files found: {len(npy_files)}")
print(f"Total OpenPose directories found: {len(json_dirs)}")

test_file = None
if npz_files:
    test_file = npz_files[0]
    print(f"\nAuditing NPZ file: {test_file}")
    with np.load(test_file) as data:
        print("Keys in file:", list(data.files))
        for key in data.files:
            print(f" - Key: '{key}', Shape: {data[key].shape}")
elif npy_files:
    test_file = npy_files[0]
    print(f"\nAuditing NPY file: {test_file}")
    data = np.load(test_file)
    print(f"Shape: {data.shape}")
elif json_dirs:
    test_dir = json_dirs[0]
    print(f"\nAuditing OpenPose Directory: {test_dir}")
    files = sorted(glob.glob(os.path.join(test_dir, "*.json")))
    print(f"Total frame files: {len(files)}")
    if files:
        import json
        with open(files[0], 'r') as f:
            data = json.load(f)
            print("Keys in JSON:", list(data.keys()))
            if 'people' in data and data['people']:
                p = data['people'][0]
                print("Keys in first person:", list(p.keys()))
else:
    print("\nNo coordinate files found. Listing directory structure:")
    for root, dirs, files in os.walk(input_dir):
        print(f"Directory: {root}")
        for file in files[:5]:
            print(f"  - {file}")


## 4. Preprocess Dataset to Compressed NPZ Format

To avoid severe CPU-disk latency bottlenecks during training, convert the raw OpenPose JSON frame files into compressed `.npz` files in parallel using all available CPU cores. This reduces step latency from ~12.5 seconds to milliseconds.

In [ ]:
# Run parallel dataset preprocessing
!python scripts/preprocess_how2sign.py

## 5. Run Model Training

This runs our streamlined end-to-end `Conformer -> T5-Small` translation pipeline using the preprocessed NPZ dataset. Adjust batch size and learning rate as needed.

In [ ]:
# --- Option A: How2Sign Training (NPZ-based) ---
import os
data_dir = "/dev/shm/how2sign_npz"

metadata_file = next((p for p in [
    '/kaggle/input/datasets/nazarboholii/how2sign/how2sign_realigned_train.csv',
    '/kaggle/input/how2sign/how2sign_realigned_train.csv',
    '/kaggle/input/how2sign-keypoints/how2sign_realigned_train.csv',
    '/kaggle/input/datasets/nazarboholii/how2sign-keypoints/how2sign_realigned_train.csv'
] if os.path.exists(p)), '/kaggle/input/datasets/nazarboholii/how2sign/how2sign_realigned_train.csv')

# Phase 1: Train model with face landmarks enabled (411 dimensions)
!python -m src.train --epochs 10 --batch_size 8 --lr 1e-4 --data_dir {data_dir} --metadata_file {metadata_file}

# Phase 2: Train model with face landmarks disabled (ablation study - 201 dimensions)
# !python -m src.train --epochs 10 --batch_size 8 --lr 1e-4 --data_dir {data_dir} --metadata_file {metadata_file} --no_face

# --- Option B: YouTube-ASL Training ---
# Phase 1: Train model with face landmarks enabled (534 dimensions)
# !python -m src.train --epochs 10 --batch_size 8 --lr 1e-4 --data_dir /kaggle/input/youtube-asl/landmarks --metadata_file /kaggle/input/youtube-asl/metadata.csv

# Phase 2: Train model with face landmarks disabled (ablation study - 258 dimensions)
# !python -m src.train --epochs 10 --batch_size 8 --lr 1e-4 --data_dir /kaggle/input/youtube-asl/landmarks --metadata_file /kaggle/input/youtube-asl/metadata.csv --no_face